# Task description
This notebook is devoted to analysis of connection between $a(t)$ and $b(t)$ 
from Ito equation: 
$$dX = a(t)dt + b(t)dW, \quad \text{where } W \text { is a normal Wiener process}$$

In [ ]:
# Import modules
import pandas as pd
import numpy as np
import plotly.express as ple
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode
from IPython.display import display, HTML
from sklearn.mixture import GaussianMixture
from tqdm.notebook import tqdm
import pickle

# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2
# Enable latex on plotly figures
init_notebook_mode()
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

## 2. Load up dataset

### 2.1. Original NASA dataset

In [ ]:
magData = pd.read_csv("../src/datasets/2020_ydhm_id.csv")
# magData.dropna(inplace=True)
magData.index = pd.DatetimeIndex(magData["ydhm_id"])
magData = magData.drop(columns=["ydhm_id"])
# magData = magData.rolling(10).mean()
# magData.dropna(inplace=True)

magData = magData[magData.index < pd.to_datetime("2020-02-01 00:00:00")]
N = magData.shape[0]
# Get subset of original dataset
dataset = magData.iloc[:N]
# dataset.index = pd.datetimeindex(dataset["ydhm_id"])
# dataset = dataset.drop(columns=["ydhm_id"])
# Free space
del magData

dataset.diff().head(20)

We have a time series. Let's consider Bx series. 
Chose window size and step. Take first one and calculate initial parameters:
p_0, a_0, b_0, a(t_0), b(t_0). 
Then we can get rid of it and move forward.

Make step (defined by window). If we've faces NaN values then sample them
by previously calculated mixture

In [ ]:
gmm = dict(
    series=dataset.diff()["Bx"].values,
    window={"size": 4200, "step": 1},
)

# Initialize model on first window and estimate parameters
model = GaussianMixture(n_components=3, covariance_type="spherical", random_state=0)
initial_window = gmm["series"][: gmm["window"]["size"]]
__initial_window = initial_window[~np.isnan(initial_window)]  # Get rid of gaps
mixture = model.fit(__initial_window.reshape(-1, 1))
# Preserving parameters
gmm["weights"] = mixture.weights_.reshape(-1, 1)
gmm["means"] = mixture.means_
gmm["variances"] = mixture.covariances_.reshape(-1, 1)
df = pd.DataFrame({"filled": np.isnan(initial_window)})
gaps_number = np.isnan(initial_window).sum()
fillings, _ = model.sample(gaps_number)
initial_window[np.isnan(initial_window)] = fillings.reshape(-1)
df["series"] = initial_window
ple.scatter(df.iloc[:2000], color="filled").show()

In [ ]:
__start = gmm["window"]["step"]
__stop = len(gmm["series"]) - gmm["window"]["size"]
__step = gmm["window"]["step"]

for i in tqdm(range(__start, __stop, __step)):
    values = gmm["series"][i : gmm["window"]["size"] + i].reshape(-1, 1)
    if any(np.isnan(values)):
        gaps_number = np.isnan(values).sum()
        fillings, _ = model.sample(gaps_number)
        values[np.isnan(values)] = fillings.reshape(-1)

    model = GaussianMixture(
        n_components=3,
        covariance_type="spherical",
        weights_init=mixture.weights_,
        means_init=mixture.means_,
        random_state=0,
    )
    mixture = model.fit(values)
    # Initialize container for parameters
    gmm["weights"] = np.append(gmm["weights"], mixture.weights_.reshape(-1, 1), axis=1)
    gmm["means"] = np.append(gmm["means"], mixture.means_, axis=1)
    gmm["variances"] = np.append(
        gmm["variances"], mixture.covariances_.reshape(-1, 1), axis=1
    )

In [ ]:
with open("saved_dictionary.pkl", "rb") as f:
    gmm = pickle.load(f)

$$
a(t) = \sum_{k=1}^{K}{p_k a_k}, \quad 
% b(t) = \sum_{k=1}^{K}{p_k\cdot(b^{2}_k + a^{2}_k) − a(t)^2}
b_1^{2}(t) = \sum_{k=1}^{K}{p_k b_k} \quad
b_2^{2}(t) = \sum_{k=1}^{K}{p_k b_k^2}
$$
$K$ is a number of mixture components

In [ ]:
p = gmm["weights"]
a = gmm["means"]
b = gmm["variances"]

coef_a = np.sum(p * a, axis=0)
# coef_b = np.sum(p * (b**2 + a**2) -coef_a **2, axis=0)
coef_b1 = np.sqrt(np.sum(p * b**2, axis=0))
coef_b2 = np.sqrt(np.sum(p * b, axis=0))

In [ ]:
gmm["coef_a"] = coef_a
gmm["coef_b1"] = coef_b1
gmm["coef_b2"] = coef_b2

In [ ]:
# import pickle

# with open('saved_dictionary.pkl', 'wb') as f:
#     pickle.dump(gmm, f)

In [ ]:
# ple.line({f"comp {i}":b[i] for i in range(p.shape[0])})

In [ ]:
# ple.line({
#     "coef_a": coef_a,
#     "coef_b": coef_b, }
# )

### Correlation plots

In [ ]:
correlation_window = (60 * 12, 1)
kernel_size = 60 * 4  # For smoothing plots

#### Correlation $a(t)$, $b_1(t)=\sqrt{\sum_{j=1}^Kp_jb_j^2}$

In [ ]:
correlation = []
for i in range(0, len(coef_a) - correlation_window[0], correlation_window[1]):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = coef_b1[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
# kernel = np.ones(kernel_size) / kernel_size
# ple.line(
#     {"correlation": correlation,
#      f"smoothed (kernel = {kernel_size})": np.convolve(correlation, kernel, mode='same',)},
#     title=f"Correlation between a(t) and b_1(t) on window size {correlation_window[0]}").show()

In [ ]:
fig = make_subplots(rows=3, cols=1)
# Dividing plot on three parts
part1, part2, part3 = [], [], []
for i, val in enumerate(correlation):
    if (i + 1) / len(correlation) < 1 / 3:
        part1.append(val)
    elif (i + 1) / len(correlation) < 2 / 3:
        part2.append(val)
    else:
        part3.append(val)
# Add plots to figure
fig.append_trace(
    go.Scatter(
        y=part1,
        name="First part",
    ),
    row=1,
    col=1,
)

fig.append_trace(
    go.Scatter(
        y=part2,
        name="Second part",
    ),
    row=2,
    col=1,
)

fig.append_trace(
    go.Scatter(
        y=part3,
        name="Third part",
    ),
    row=3,
    col=1,
)

title = (
    r"""
$\text{Correlation between } a(t), b_1(t)=\sqrt{\sum_{j=1}^Kp_jb_j^2} \text{ on window size }
"""
    + f"{correlation_window[0]}$"
)

fig.update_layout(height=800, width=1200, title_text=title)
fig.show()

#### Correlation $a(t)$, $b_1^2(t)$

In [ ]:
correlation = []
for i in range(0, len(coef_a) - correlation_window[0], correlation_window[1]):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = (coef_b1**2)[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
# kernel = np.ones(kernel_size) / kernel_size
# ple.line(
#     {"correlation": correlation,
#      f"smoothed (kernel = {kernel_size})": np.convolve(correlation, kernel, mode='same',)},
#     title=f"Correlation between a(t) and b_1^2(t) on window size {correlation_window[0]}").show()

In [ ]:
fig = make_subplots(rows=3, cols=1)
# Dividing plot on three parts
part1, part2, part3 = [], [], []
for i, val in enumerate(correlation):
    if (i + 1) / len(correlation) < 1 / 3:
        part1.append(val)
    elif (i + 1) / len(correlation) < 2 / 3:
        part2.append(val)
    else:
        part3.append(val)
# Add plots to figure
fig.append_trace(
    go.Scatter(
        y=part1,
        name="First part",
    ),
    row=1,
    col=1,
)

fig.append_trace(
    go.Scatter(
        y=part2,
        name="Second part",
    ),
    row=2,
    col=1,
)

fig.append_trace(
    go.Scatter(
        y=part3,
        name="Third part",
    ),
    row=3,
    col=1,
)

title = (
    r"""
$\text{Correlation between } a(t), b_1^2(t)=\sum_{j=1}^Kp_jb_j^2 \text{ on window size }
"""
    + f"{correlation_window[0]}$"
)

fig.update_layout(height=800, width=1200, title_text=title)
fig.show()

#### Correlation $a(t)$, $b_2(t)=\sqrt{\sum_{j=1}^Kp_jb_j}$

In [ ]:
correlation = []
correlation_window = (60 * 12, 1)
for i in range(0, len(coef_a) - correlation_window[0], correlation_window[1]):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = coef_b2[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
# kernel_size = 60*4
# kernel = np.ones(kernel_size) / kernel_size
# ple.line(
#     {"correlation": correlation,
#      f"smoothed (kernel = {kernel_size})": np.convolve(correlation, kernel, mode='same',)},
#     title=f"Correlation between a(t) and b_2(t) on window size {correlation_window[0]}").show()

In [ ]:
fig = make_subplots(rows=3, cols=1)
# Dividing plot on three parts
part1, part2, part3 = [], [], []
for i, val in enumerate(correlation):
    if (i + 1) / len(correlation) < 1 / 3:
        part1.append(val)
    elif (i + 1) / len(correlation) < 2 / 3:
        part2.append(val)
    else:
        part3.append(val)
# Add plots to figure
fig.append_trace(
    go.Scatter(
        y=part1,
        name="First part",
    ),
    row=1,
    col=1,
)

fig.append_trace(
    go.Scatter(
        y=part2,
        name="Second part",
    ),
    row=2,
    col=1,
)

fig.append_trace(
    go.Scatter(
        y=part3,
        name="Third part",
    ),
    row=3,
    col=1,
)

title = (
    r"""
$\text{Correlation between } a(t), b_2^2(t)=\sqrt{\sum_{j=1}^Kp_jb_j} \text{ on window size }
"""
    + f"{correlation_window[0]}$"
)

fig.update_layout(height=800, width=1200, title_text=title)
fig.show()

#### Correlation $a(t)$, $b_2^2(t)$

In [ ]:
correlation = []
correlation_window = (60 * 12, 1)
for i in range(0, len(coef_a) - correlation_window[0], correlation_window[1]):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = (coef_b2**2)[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
# kernel_size = 60*4
# kernel = np.ones(kernel_size) / kernel_size
# ple.line(
#     {"correlation": correlation,
#      f"smoothed (kernel = {kernel_size})": np.convolve(correlation, kernel, mode='same',)},
#     title=f"Correlation between a(t) and b_2^2(t) on window size {correlation_window[0]}").show()

In [ ]:
fig = make_subplots(rows=3, cols=1)
# Dividing plot on three parts
part1, part2, part3 = [], [], []
for i, val in enumerate(correlation):
    if (i + 1) / len(correlation) < 1 / 3:
        part1.append(val)
    elif (i + 1) / len(correlation) < 2 / 3:
        part2.append(val)
    else:
        part3.append(val)
# Add plots to figure
fig.append_trace(
    go.Scatter(
        y=part1,
        name="First part",
    ),
    row=1,
    col=1,
)

fig.append_trace(
    go.Scatter(
        y=part2,
        name="Second part",
    ),
    row=2,
    col=1,
)

fig.append_trace(
    go.Scatter(
        y=part3,
        name="Third part",
    ),
    row=3,
    col=1,
)

title = (
    r"""
$\text{Correlation between } a(t), b_2^2(t)=\sum_{j=1}^Kp_jb_j \text{ on window size }
"""
    + f"{correlation_window[0]}$"
)

fig.update_layout(height=800, width=1200, title_text=title)
fig.show()